# Demo: ledger2 transactions

How does a fact get into the ledger, and what does it mean once it
is there? Every write is one `with L.transaction(by=...)` block that
becomes one revision; every row is `(subject, predicate, object)`;
and the predicate's cardinality decides whether a second `add`
replaces the first or joins it. This notebook writes a few facts
about game G5 and the `normie` game set and reads them back at each
revision.

In [1]:
import os, shutil, tempfile, time
from pathlib import Path

from dvc_dat.ledger2 import Ledger, LockTimeout

ledger_dir = Path(tempfile.mkdtemp(prefix="demo-l2-tx-"))
os.environ["DAT_LEDGER"] = str(ledger_dir)
L = Ledger.open()
print(L.at(), L.at().all())

Rev(0) []


`Ledger.open()` reads `DAT_LEDGER`. An empty ledger is revision 0;
`L.at()` is a frozen view of the top, and `all()` lists every row.

## One block, one revision

In [2]:
with L.transaction(by="human:dan") as t:
    t.add("game/G5", "court", "hs-wood-2")
    t.add("game/G5", "season", "2026")
print(t.rev)
for row in L.at().all():
    print(row)

Rev(1)
('ledger/tx/1', 'by', 'human:dan')
('game/G5', 'court', 'hs-wood-2')
('game/G5', 'season', '2026')


Two `add`s became revision 1. The block also wrote a third row:
`by` on the transaction's own subject, `ledger/tx/1`.

## Keywords are predicates on the transaction

In [3]:
with L.transaction(by="agent:svp", time=True,
                   why="re-cut", ticket="T531") as t:
    t.add("game/G5", "venue", "Lincoln HS")
    t.add(t.tx, "checked", "G1,G4,G5")
print("inside the block t.tx was a placeholder:", t.tx[:22], "...")
for p, o in L.at().all(t.rev.tx):
    print(f"  {t.rev.tx}  {p:8} {o}")

inside the block t.tx was a placeholder: ledger/tx/pending-3d5b ...
  ledger/tx/2  by       agent:svp
  ledger/tx/2  time     2026-10-18T09:24:28.867595+00:00
  ledger/tx/2  why      re-cut
  ledger/tx/2  ticket   T531
  ledger/tx/2  checked  G1,G4,G5


`time=True` stamps the UTC clock; `why` and `ticket` are any
keywords; `t.tx` names the same subject for a row decided inside the
body, rewritten to `ledger/tx/2` at commit.

In [4]:
rev = L.at()
print(rev["game/G5", "court"])
print(rev.get("game/G5", "referee", default="(none)"))
print(rev[rev.tx, "by"])
print(rev.refs(s="ledger/tx/*", where={"ticket": "T531"}))

hs-wood-2
(none)
agent:svp
[Ref(2, 'ledger/tx/2')]


Provenance reads like any row: `[]` is exactly one match or raise,
`get` at most one, and `refs` finds the transaction by its keyword.

## A second add on a `one` predicate replaces; `remove` clears

In [5]:
with L.transaction(by="human:dan") as t:
    t.add("game/G5", "court", "hs-wood-3")
with L.transaction(by="human:dan") as t:
    t.remove("game/G5", "court", "hs-wood-3")
for r in (L.at(1), L.at(3), L.at()):
    print(r, r.get("game/G5", "court"))
print("asserted at:", L.at()["game/G5", "court", "hs-wood-2"])

Rev(1) hs-wood-2
Rev(3) hs-wood-3
Rev(4) None
asserted at: Rev(1)


Every predicate is `one` unless declared otherwise: the latest
`add` wins, and a `remove` of the held value leaves none. Old
revisions still read the old values, and a three-position pattern
returns the revision that asserted the triple.

## Declaring a predicate `many`

In [6]:
with L.transaction(by="human:dan") as t:
    t.add("gameset/normie", "contains", "game/G1")
    t.add("gameset/normie", "contains", "game/G4")
before = L.at()
print("one: ", before.all("gameset/normie", "contains"))

with L.transaction(by="human:dan") as t:
    t.add("ledger/pred/contains", "cardinality", "many")
    t.add("ledger/pred/contains", "doc", "a set's members")
with L.transaction(by="human:dan") as t:
    t.add("gameset/normie", "contains", "game/G1")
    t.add("gameset/normie", "contains", "game/G5")
print("many:", L.at().all("gameset/normie", "contains"))
print("before, still:", before.all("gameset/normie", "contains"))

one:  ['game/G4']
many: ['game/G4', 'game/G1', 'game/G5']
before, still: ['game/G4']


Cardinality is itself rows, on the subject `ledger/pred/contains`.
Under `one` the second `add` replaced G1; once `contains` is `many`
the adds accumulate. The old view `before` keeps reading `one`.

In [7]:
with L.transaction(by="human:dan") as t:
    t.remove("gameset/normie", "contains", "game/G4")
rev = L.at()
print(rev.all("gameset/normie", "contains"))
print(("gameset/normie", "contains", "game/G4") in rev)
try:
    rev["gameset/normie", "contains"]
except LookupError as e:
    print(type(e).__name__, "-", e)

['game/G1', 'game/G5']
False
Multiple - 2 matches for ('gameset/normie', 'contains') at rev 8


On a `many` predicate `remove` takes one member out. `in` asks
about one triple; `[]` refuses a pattern with two matches.

## `lock=`: read and write against a top that cannot move

In [8]:
with L.transaction(by="agent:svp", lock=5) as t:
    season = t.base["game/G5", "season"]
    t.add("game/G5", "season", str(int(season) + 1))
print(L.at()["game/G5", "season"])

2027


Under `lock=5` the block holds the write lock for at most 5 s, so
`t.base` stays the top until exit: the read and the write cannot be
split by another writer.

In [9]:
top = L.at().id
try:
    with L.transaction(by="agent:svp", lock=0.2) as t:
        t.add("game/G5", "season", "1999")
        time.sleep(2)
except LockTimeout as e:
    print("LockTimeout:", e)
print("revision unchanged:", L.at().id == top)

LockTimeout: lock=0.2: the body ran past its limit; nothing written
revision unchanged: True


A body that runs past its limit is interrupted and writes nothing.

## What a transaction refuses

In [10]:
def refused(f):
    try:
        f()
        return "not refused"
    except Exception as e:
        return f"{type(e).__name__}: {e}"


def add_int():
    with L.transaction(by="human:dan") as t:
        t.add("game/G5", "period", 3)


def raises_inside():
    with L.transaction(by="human:dan") as t:
        t.add("game/G5", "court", "outdoor-1")
        raise RuntimeError("vocabulary check failed")


bare = L.transaction(by="human:dan")
top = L.at().id
print(refused(add_int))
print(refused(lambda: bare.add("game/G5", "court", "x")))
print(refused(raises_inside))
print("revision unchanged:", L.at().id == top)

ValueError: a ledger object is a str, a JSON dict or list, or a Value/Ref; not int (3): a number is its string (Q5), say str(3)
RuntimeError: transaction is new: add goes inside `with L.transaction(...) as t:`
RuntimeError: vocabulary check failed
revision unchanged: True


A number is written as its string (`"3"`); a transaction is used
only as `with`; and an exception inside the block writes nothing.

In [11]:
shutil.rmtree(ledger_dir)
del os.environ["DAT_LEDGER"]
print("removed:", not ledger_dir.exists())

removed: True
